# Node · 4-hour Colab conversation experiment

Continue the existing Node checkpoint on a Colab CPU runtime. **No pretrained model is substituted.** CPU execution is requested explicitly; no GPU allocation, purchase, or upgrade is needed.

This version also keeps the **best conversational checkpoint automatically**, using rule-based conversation checks, with baseline held-out-loss and per-case regression guards, instead of assuming the last step is best.

1. Open this notebook in Google Colab.
2. Run the cells in order. The training stage lasts **14,400 seconds**; before/after evaluation adds time.
3. If Colab terminates the runtime early, the run stops there and local files may be lost. A browser disconnect does not necessarily stop execution. This notebook does not mount Google Drive.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os
import torch

print("PyTorch:", torch.__version__)
print("Execution device: CPU (explicitly selected)")
print("Available CPU cores:", os.cpu_count())
# Colab's existing PyTorch can run CPU operations; do not reinstall it unnecessarily.


In [ ]:
ROOT = Path("/content/node-ai-colab")
REPO = "https://github.com/ItzJoker2YT-city/node-ai.git"
if ROOT.exists():
    raise RuntimeError("This project folder already exists. Reuse your existing run or choose a new empty folder; nothing was overwritten.")
subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
assert (ROOT / "colab/train_colab.py").is_file()
print("Source commit:", subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip())


## Run the 4-hour experiment
The cell below performs real CPU training. Logs show elapsed seconds, steps, assigned hardware, held-out validation loss, and conversation-check scores. A baseline is saved before any update. Checkpoints are stored in the Colab runtime, **not Google Drive**.

Do not rerun this cell while it is still executing. No GPU or paid compute is requested by this notebook.


In [ ]:
command = [sys.executable, str(ROOT / "colab/train_colab.py"),
           "--seconds", "14400", "--device", "cpu", "--batch-size", "2"]
subprocess.run(command, cwd=ROOT, check=True)


In [ ]:
OUTPUT = ROOT / "colab_outputs"
report = json.loads((OUTPUT / "evaluation.json").read_text())
print(json.dumps(report["run"], indent=2))
print("\n=== baseline conversation checks ===")
for item in report["before_quality"]["cases"]:
    print(f"[{item['score']}/{item['max_score']}] {item['case']}")
    print(item['output'])
    print()
print("=== best checkpoint conversation checks ===")
for item in report["after_best_quality"]["cases"]:
    print(f"[{item['score']}/{item['max_score']}] {item['case']}")
    print(item['output'])
    print()


## Download the results
`best.pt` is the strongest eligible checkpoint by rule-based conversation score, then held-out loss. A candidate is rejected if its loss exceeds the baseline or any baseline conversation case loses points. These checks cannot guarantee general conversation quality. `final.pt` is the end-of-run checkpoint. Copy the chosen checkpoint to your local project's `data/node.pt` **after backing up the old file and stopping the local server**. These weights are CPU-portable.

Even with a longer run, read `evaluation.json` before deciding which weights to keep. The best checkpoint can still be the unchanged original baseline if training quality drops.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/node-colab-results", "zip", OUTPUT)
print("Created:", archive)
files.download(archive)


## Release the runtime
After the ZIP has finished downloading, use **Runtime → Disconnect and delete runtime**. The repository and notebook do not create a public chat server, mount Drive, request secrets, or upload your Google account details to GitHub.